# Topic: Advanced Prompt Engineering (Zero-shot, Few-shot, CoT, JSON, Caching)

## Definition (30-second explanation)
Think of an LLM as a brilliant but highly literal intern. **Zero-shot** is handing them a task with no examples. **Few-shot** is giving them a template of previous good work to mimic. **Chain-of-Thought (CoT)** is forcing them to write out their logic on a scratchpad before giving you the final answer. **JSON Formatting** is handing them a strict fill-in-the-blank form so your downstream software can actually read their work.

## Why Interviewers Ask This
In applied GenAI, you don't fine-tune a model for every single problem—it's too expensive. Interviewers want to know if you can manipulate a frozen model's behavior reliably for production pipelines. A candidate who knows how to force an LLM to output perfect JSON, and knows when to use CoT vs. Prompt Caching, can build production apps tomorrow.

## Core Concepts (The 3-Layer Anatomy)
*   **The Bottleneck:** LLMs naturally output conversational, unstructured text ("chat"). In software engineering, unstructured text breaks downstream APIs. Furthermore, complex reasoning tasks fail if the model tries to answer immediately.
*   **The Mechanism:**
    *   **Few-Shot:** Uses the context window to condition the model's activations (In-Context Learning) without updating its actual weights. 
    *   **Chain-of-Thought (CoT):** LLMs generate tokens autoregressively. CoT forces the model to generate intermediate "thinking" tokens. Since compute happens *per token*, more tokens = more compute time to arrive at the correct answer.
    *   **Output Formatting:** Uses strict system prompts and API-level schema enforcers (like OpenAI's JSON mode or LangChain's Pydantic parsers) to constrain the model's vocabulary during generation.
    *   **Prompt Caching:** Temporarily stores the Key-Value (KV) cache of massive system prompts on the GPU so subsequent API calls don't have to recompute the same input tokens, slashing Time-to-First-Token (TTFT) and cost.
*   **The Trade-off:** CoT improves accuracy but directly increases latency and token costs. Few-shot consumes context window space and increases input token costs unless prompt caching is utilized.

## When to Use
*   **Zero-shot:** Simple summarization, translation, or basic sentiment extraction.
*   **Few-shot:** Style matching, complex classification, or teaching the model a custom domain-specific output format.
*   **CoT:** Math, logic puzzles, Text-to-SQL, or multi-step reasoning.
*   **Prompt Caching:** Large, static system prompts (e.g., passing a massive database schema into every Text-to-SQL prompt).

## Advantages
*   Zero training cost. You can change model behavior instantly in production.
*   JSON mode allows LLMs to act as deterministic software components (e.g., returning `{"sql": "SELECT *", "confidence": 0.9}`).

## Limitations
*   **Prompt Fragility:** A prompt that works on GPT-4 might break on Llama-3.
*   **Context Limits:** You can only fit so many Few-Shot examples before hitting context windows or losing the model's attention (the "Lost in the Middle" phenomenon).

## Common Comparisons
*   **Prompting vs. Fine-Tuning:** Prompting teaches *behavior* and *format* (in-context). Fine-Tuning teaches new *domain knowledge* and *permanent tone*.
*   **Zero-shot vs. Few-shot:** Zero-shot is fast/cheap but unreliable for formatting. Few-shot is highly reliable but costs more input tokens.

## Common Interview Traps
*   **The "Hallucination Fix" Trap:** Candidates often suggest Prompting/CoT to fix a model lacking factual knowledge. Prompting cannot teach a model facts it never learned; that requires RAG (Retrieval-Augmented Generation) or Fine-Tuning.
*   **Ignoring Token Costs in CoT:** If you say you will use CoT for a high-volume pipeline, and the interviewer asks about latency, you *must* mention that generating reasoning tokens slows down the system.

## Python Syntax (LangChain with Pydantic JSON Output)
```python
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

# 1. Define the exact JSON structure you want the LLM to output
class SQLQuery(BaseModel):
    query: str = Field(description="The executable SQL query")
    reasoning: str = Field(description="Step-by-step reasoning for the query")

# 2. Create the parser to enforce the schema
parser = PydanticOutputParser(pydantic_object=SQLQuery)

# 3. Inject the formatting instructions directly into the prompt
prompt = PromptTemplate(
    template="Write a SQL query for this request: {request}.\n\n{format_instructions}",
    input_variables=["request"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# In a LangChain pipeline, passing the output to 'parser' guarantees a Python dictionary/JSON
# chain = prompt | llm | parser 
```

## 45-Second Interview Answer
"In production GenAI, advanced prompting is about control and reliability. I use Few-Shot prompting to condition the model's output structure without the cost of fine-tuning. For complex logic, like Text-to-SQL, I inject Chain-of-Thought instructions to force the model to generate intermediate reasoning tokens, which unlocks its logical capabilities at the cost of slightly higher output latency. Finally, to ensure the LLM can integrate into a traditional software pipeline, I use tools like Pydantic parsers to strictly enforce JSON output schemas, preventing conversational hallucinations from breaking downstream API execution."

## Practice Questions:

### Q1: Production Text-to-SQL & CoT Mechanics
**Question:** Your Text-to-SQL pipeline is hallucinating table names and returning conversational text instead of raw SQL. How do you fix this with Few-Shot prompting and JSON formatting? Technically, why does adding Chain-of-Thought (CoT) fix complex logic, and what is the trade-off?

**Answer:**
1. **The Fix (Few-Shot & JSON):** 
   * First, I define a strict Pydantic BaseModel for my output (e.g., `{"thought_process": "...", "query": "..."}`). I pass this directly into the LLM's API (like OpenAI's `response_format`) to guarantee the downstream output is machine-parseable.
   * To fix hallucinations, I inject the database schema into the system prompt (which I would Cache to save on input token costs). Then, I apply **Few-Shot Prompting**: I provide 3 to 5 perfect examples of a user question mapping to a valid JSON SQL response. This conditions the model's behavior to follow my strict schema rules.
2. **CoT Mechanics & Trade-offs:** 
   * LLMs do not have hidden compute cycles to "think" before they speak; their only compute mechanism is generating the next token. If forced to output the final SQL immediately, they often fail on complex joins. CoT forces the model to generate intermediate reasoning tokens first, effectively "buying compute time" to organize its logic before generating the final query. 
   * **The Trade-off:** Every intermediate "thinking" token generated costs money and takes time. CoT significantly increases output token costs and increases Time-to-Final-Answer (latency), so it should only be used for complex logical routing, not simple extraction.

**Interview Tips:**
*   **The Modern Flex:** Mentioning modern API SDK features like OpenAI's `response_format` and `.parsed` proves you are a practitioner, not just a theorist. 
*   **The "Compute Time" Analogy:** Explaining that "generating tokens = compute time" is the exact mental model senior GenAI engineers use.